# FMCG Demand Forecasting & Product Intelligence — EDA

Exploratory analysis on the real FMCG distribution dataset (2022-2024) used by the platform.

- 30 SKUs across 5 categories (Milk, Yogurt, ReadyMeal, Juice, SnackBar)
- 3 channels (Retail, Discount, E-commerce), 3 regions (PL-Central, PL-North, PL-South)
- 190,757 daily fact rows; 31,027 weekly modeling rows

Negative `units_sold` values are valid (returns) and are preserved throughout.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if (Path.cwd().name == 'notebooks') else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.analytics import eda
from src.models import feature_engineering

RAW = ROOT / 'data' / 'raw'
daily_df = pd.read_csv(RAW / 'FMCG_2022_2024.csv')
weekly_df = pd.read_csv(RAW / 'weekly_df_final_for_modeling.csv')
enriched_df = pd.read_csv(RAW / 'df_weekly_MI-006_enriched.csv')

daily_df['date'] = pd.to_datetime(daily_df['date'])
weekly_df['week'] = pd.to_datetime(weekly_df['week'])
category_lookup = daily_df[['sku', 'category']].drop_duplicates().set_index('sku')['category']
weekly_df['category'] = weekly_df['sku'].map(category_lookup)

print(f"daily rows: {len(daily_df):,}")
print(f"weekly rows: {len(weekly_df):,}")
print(f"unique SKUs: {weekly_df['sku'].nunique()}")
print(f"categories: {sorted(weekly_df['category'].dropna().unique())}")

## 1. Weekly sales trend by category

In [ ]:
by_category = eda.sales_by_category(weekly_df, period='weekly')
fig, ax = plt.subplots(figsize=(12, 5))
for category, group in by_category.groupby('category'):
    ax.plot(group['period'], group['units_sold'], label=category, linewidth=1.5)
ax.set_title('Weekly units_sold by category (2022-2024)')
ax.set_xlabel('Week')
ax.set_ylabel('Units sold')
ax.legend(loc='upper left')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 2. Channel comparison

In [ ]:
summary = eda.channel_comparison(daily_df)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(summary['channel'], summary['total_units_sold'])
axes[0].set_title('Total units sold by channel')
axes[0].set_ylabel('Units sold')
axes[1].bar(summary['channel'], summary['promo_lift_pct'])
axes[1].set_title('Promo lift (%) by channel')
axes[1].set_ylabel('Promo lift %')
plt.tight_layout()
plt.show()
summary

## 3. Regional heatmap (units sold)

In [ ]:
pivot = daily_df.pivot_table(index='region', columns='category', values='units_sold', aggfunc='sum')
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlGnBu')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=30)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title('Total units_sold by region x category')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 4. Promotion impact

In [ ]:
impact = eda.promo_impact_analysis(weekly_df).sort_values('promo_lift_pct', ascending=False)
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(impact['sku'], impact['promo_lift_pct'])
ax.set_title('Promo lift % per SKU (sorted)')
ax.set_ylabel('Promo lift %')
ax.set_xticklabels(impact['sku'], rotation=70)
plt.tight_layout()
plt.show()
impact.head(10)

## 5. Lifecycle stage distribution

In [ ]:
distribution = eda.lifecycle_distribution(weekly_df)
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(distribution.keys(), distribution.values(), color=['#4caf50', '#2196f3', '#ff7043'])
ax.set_title('SKU lifecycle distribution')
ax.set_ylabel('SKU count')
for i, value in enumerate(distribution.values()):
    ax.text(i, value + 0.1, str(value), ha='center')
plt.tight_layout()
plt.show()
distribution

## 6. Seasonality (holiday / summer / winter effects)

In [ ]:
season_df = weekly_df.assign(
    season=lambda f: f.apply(
        lambda row: 'holiday' if row['is_holiday_week'] == 1 else (
            'summer' if row['is_summer'] == 1 else (
                'winter' if row['is_winter'] == 1 else 'shoulder'
            )
        ),
        axis=1,
    )
)
season_means = season_df.groupby('season')['units_sold'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
season_means.plot(kind='bar', ax=ax, color='#5c6bc0')
ax.set_title('Mean weekly units_sold by season flag')
ax.set_ylabel('Mean units_sold')
plt.tight_layout()
plt.show()
season_means

## 7. Feature correlation heatmap

In [ ]:
features, _target = feature_engineering.build_training_features(weekly_df)
corr = eda.correlation_matrix(features)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=70)
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
ax.set_title('Feature correlation matrix')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 8. GBR vs Ridge — model comparison

Walk-forward CV emits MAPE / RMSE / MAE / R² for each model. The framework returns a winner with rationale: if GBR's MAPE improvement < 5%, Ridge wins on simplicity.

In [ ]:
from src.models import evaluation
report = evaluation.compare_models(weekly_df.head(5000))  # subset for notebook speed
comparison = pd.DataFrame({
    'gbr': report.gbr_metrics,
    'ridge': report.ridge_metrics,
})
print(f"winner: {report.winner}")
print(f"rationale: {report.rationale}")
comparison